In [3]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.discriminant_analysis import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import compute_sample_weight
from tqdm import tqdm
from ibl_info.decoder_pid import compute_decoder_pid
from ibl_info.decoder_utils import load_specific_regions
from ibl_info.dual_decoders import complete_decoder_pid_with_null, compute_null_distribution
from ibl_info.prepare_data_pid import get_new_cinc_intervals, get_new_cinc_intervals_choice
from ibl_info.utils import epoch_events
from one.api import ONE
from prior_localization.prepare_data import prepare_widefield
from brainbox.io.one import SessionLoader
from brainwidemap.bwm_loading import load_trials_and_mask
import numpy as np
import pickle as pkl
import os
from joblib import Parallel, delayed
from communication_subspace.ibl_communication.utils import (
    check_config,
    compute_reduced_rank_pairs,
    compute_regionwise_r2,
    get_high_low_masks,
    get_intrinsic_dimensions,
    load_widefield_epoch,
)

config = check_config()
from one.api import ONE
from pathlib import Path
import yaml
import os
import wfield
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from prior_localization.prepare_data import prepare_widefield
from brainbox.io.one import SessionLoader
from brainwidemap.bwm_loading import load_trials_and_mask
from prior_localization.functions.utils import compute_mask
from scipy import stats
import pickle as pkl
from scipy.spatial import KDTree

In [ ]:
%load_ext autoreload
%autoreload 2

In [60]:
def build_candidate_pools(df, feature_cols, k_neighbors=20):
    """
    Builds a matrix of candidate trial indices for null distributions.

    Args:
        df: Pandas DataFrame containing trial information.
        feature_cols: List of column names to use for distance (e.g., ['sign_cont', 'prior']).
        k_neighbors: Number of candidate trials to pool per trial.

    Returns:
        candidate_pools: (N_trials, k_neighbors) integer array of trial indices.
    """

    features = df[feature_cols].values.astype(float)

    # min max transform, but is it necessary?
    min_vals = np.min(features, axis=0)
    max_vals = np.max(features, axis=0)

    range_vals = np.where((max_vals - min_vals) == 0, 1, max_vals - min_vals)
    features_norm = (features - min_vals) / range_vals

    tree = KDTree(features_norm)
    _, indices = tree.query(features_norm, k=k_neighbors + 1)

    candidate_pools = indices[:, 1:]  # type: ignore

    return candidate_pools

In [64]:
alleidpmotivation = pkl.load(
    open("../../public_hybrid_rnn/session_data/wifimicemotivation.pkl", "rb")
)
alleidactionkernels = pkl.load(
    open("../data/processed/all_eids_dict_single_zeta_complete_wifi.pkl", "rb")
)
modified_trials = pkl.load(open("../data/processed/wifi_trials_df_all.pkl", "rb"))
# modified_trials = {}
# for k in alleidactionkernels.keys():
#     trials_df = alleidactionkernels[k].copy()
#     out = np.nan_to_num(trials_df.contrastLeft.values) - np.nan_to_num(
#         trials_df.contrastRight.values
#     )
#     trials_df["sign_cont"] = out
#     trials_df["motivation"] = alleidpmotivation[str(k)]
#     modified_trials[str(k)] = trials_df
# with open("../data/processed/wifi_trials_df_all.pkl", "wb") as f:
#     pkl.dump(modified_trials, f)

In [65]:
one = ONE()
sessions = one.search(datasets="widefieldU.images.npy")
session_id = sessions[0]
engagement_signal = alleidpmotivation[str(session_id)]

In [74]:
from communication_subspace.ibl_communication.utils import (
    build_candidate_pools,
    check_config,
    compute_reduced_rank_pairs,
    compute_regionwise_null_r2,
    compute_regionwise_r2,
    generate_pseudosessions,
    get_high_low_masks,
    get_intrinsic_dimensions,
    load_widefield_epoch,
    setup_logger,
)

config = check_config()
logger = setup_logger("RidgeRegressors")

In [75]:
one = ONE(
    # base_url="https://openalyx.internationalbrainlab.org",
    # password="international",
    # silent=True,
    # username="intbrainlab",
    mode="local",
)
logger.info("trials are loaded")
trials, mask = load_trials_and_mask(
    one,
    session_id,
    exclude_nochoice=True,
    exclude_unbiased=False,
)
complete_df = modified_trials[str(session_id)]
# trials = trials[mask]
trials = complete_df[mask]
candidate_trials = build_candidate_pools(trials, ["sign_cont", "prior"])
pseudosession_matrix = generate_pseudosessions(candidate_trials, n_pseudosessions=5)

2026-05-19 15:19:32 [INFO] trials are loaded


In [78]:
logger.info(f"Loading stimulus data for {session_id}")

stimulus_data, region_names_stim = load_widefield_epoch(
    one, session_id, trials, config["hemisphere"], epoch="stim"
)

logger.info(f"Loading choice data for {session_id}")
choice_data, region_names_choice = load_widefield_epoch(
    one, session_id, trials, config["hemisphere"], epoch="choice"
)

In [79]:
assert region_names_stim == region_names_choice

# 1. find regions with significant transmission
# just use two frames
frame_idx = 0
frame_idy = 1

In [ ]:
null_results = compute_regionwise_null_r2(
    stimulus_data,
    choice_data,
    frame_idx,
    frame_idy,
    candidate_trials_matrix=pseudosession_matrix,
    n_iterations=pseudosession_matrix.shape[1],
)

In [ ]:
from statsmodels.stats.multitest import multipletests

In [88]:
# now we compare


def find_significant_pairs(true_r2, null_r2, alpha=0.05):
    n_regions = true_r2.shape[0]
    n_perms = null_r2.shape[2]

    off_diag_mask = ~np.eye(n_regions, dtype=bool)
    valid_true = true_r2[off_diag_mask]
    valid_null = null_r2[off_diag_mask]

    exceedances = np.sum(valid_null >= valid_true[:, None], axis=1)
    p_values_raw = (exceedances + 1) / (n_perms + 1)

    reject, p_values_corrected, _, _ = multipletests(p_values_raw, alpha=alpha, method="fdr_bh")

    sig_matrix = np.zeros((n_regions, n_regions), dtype=bool)
    p_matrix_adj = np.ones((n_regions, n_regions))

    sig_matrix[off_diag_mask] = reject
    p_matrix_adj[off_diag_mask] = p_values_corrected

    return sig_matrix, p_matrix_adj